In [1]:
# Stage 2 artifacts 복원
import pickle
from pathlib import Path

artifacts_path = Path('artifacts/stage2_data.pkl')

if not artifacts_path.exists():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    import shutil
    artifacts_path.parent.mkdir(exist_ok=True)
    shutil.copy('/content/drive/MyDrive/bakerstreet_artifacts/stage2_data.pkl', artifacts_path)

with open(artifacts_path, 'rb') as f:
    stage2 = pickle.load(f)

raw_store = stage2['raw_store']
shared_attrs = stage2['shared_attrs']
account_to_jurisdiction = stage2['account_to_jurisdiction']
account_to_entity_node_id = stage2['account_to_entity_node_id']
cycle_demo = stage2['cycle_demo']
fanin_demo = stage2['fanin_demo']
fanout_demo = stage2['fanout_demo']

# entity 식별자 매핑 (편의용)
entity_id_to_account = {f"ent_{nid}": acct for acct, nid in account_to_entity_node_id.items()}
account_to_entity_id = {acct: f"ent_{nid}" for acct, nid in account_to_entity_node_id.items()}

print(f"Stage 2 artifacts loaded.")
print(f"  raw_store: {len(raw_store)} records")
print(f"  - transactions: {sum(1 for r in raw_store.values() if r['type'] == 'transaction')}")
print(f"  - accounts:     {sum(1 for r in raw_store.values() if r['type'] == 'account')}")
print(f"  - registrations: {sum(1 for r in raw_store.values() if r['type'] == 'registration')}")
print(f"  - ip_logs:      {sum(1 for r in raw_store.values() if r['type'] == 'ip')}")

Mounted at /content/drive
Stage 2 artifacts loaded.
  raw_store: 35 records
  - transactions: 13
  - accounts:     15
  - registrations: 6
  - ip_logs:      1


In [2]:
# Cycle detection — DFS on entity-level directed graph

from collections import defaultdict

def build_flow_graph(raw_store, account_to_entity_id):
    """
    transaction RawRecord들을 entity-level directed graph로 변환.
    같은 entity의 여러 계좌 → 하나의 노드로 합침.
    """
    graph = defaultdict(list)  # from_entity → [(to_entity, tx_id), ...]

    for rid, rec in raw_store.items():
        if rec['type'] != 'transaction':
            continue
        sender_acct = rec['payload']['sender_account']
        receiver_acct = rec['payload']['receiver_account']
        sender_ent = account_to_entity_id.get(sender_acct)
        receiver_ent = account_to_entity_id.get(receiver_acct)
        if sender_ent and receiver_ent:
            graph[sender_ent].append((receiver_ent, rid))

    return graph


def find_cycles(graph, max_length=10):
    """
    Directed graph에서 simple cycle을 모두 찾음.
    각 cycle: 노드 리스트 (시작 == 끝)와 transaction id 리스트.
    """
    cycles = []

    def dfs(start_node, current_node, path_nodes, path_edges, visited_in_path):
        if len(path_nodes) > max_length:
            return
        for next_node, tx_id in graph.get(current_node, []):
            if next_node == start_node and len(path_nodes) >= 2:
                # cycle 닫힘
                cycles.append({
                    'nodes': path_nodes + [start_node],
                    'tx_ids': path_edges + [tx_id],
                })
            elif next_node not in visited_in_path:
                dfs(start_node,
                    next_node,
                    path_nodes + [next_node],
                    path_edges + [tx_id],
                    visited_in_path | {next_node})

    for start in graph:
        dfs(start, start, [start], [], {start})

    # 같은 cycle을 다른 시작점에서 발견하는 중복 제거
    unique_cycles = []
    seen = set()
    for c in cycles:
        # cycle을 정규화: 가장 작은 노드 id로 시작하도록 회전
        nodes = c['nodes'][:-1]  # 마지막 == 시작이라 제거
        min_idx = nodes.index(min(nodes))
        rotated = tuple(nodes[min_idx:] + nodes[:min_idx])
        if rotated not in seen:
            seen.add(rotated)
            unique_cycles.append(c)

    return unique_cycles


# 실행
flow_graph = build_flow_graph(raw_store, account_to_entity_id)
detected_cycles = find_cycles(flow_graph)

print(f"=== Flow graph ===")
print(f"  Nodes (entities): {len(flow_graph)}")
total_edges = sum(len(v) for v in flow_graph.values())
print(f"  Edges (transactions): {total_edges}")

print(f"\n=== Detected cycles: {len(detected_cycles)} ===")
for i, c in enumerate(detected_cycles):
    print(f"\nCycle {i+1} (length {len(c['nodes'])-1}):")
    for j, node in enumerate(c['nodes'][:-1]):
        next_node = c['nodes'][j+1]
        tx_id = c['tx_ids'][j]
        print(f"  {node} → {next_node}  ({tx_id})")

=== Flow graph ===
  Nodes (entities): 10
  Edges (transactions): 13

=== Detected cycles: 1 ===

Cycle 1 (length 6):
  ent_55037966 → ent_20132624  (tx_cycle_01)
  ent_20132624 → ent_10171875  (tx_cycle_02)
  ent_10171875 → ent_240240462  (tx_cycle_03)
  ent_240240462 → ent_240247005  (tx_cycle_04)
  ent_240247005 → ent_10055649  (tx_cycle_05)
  ent_10055649 → ent_55037966  (tx_cycle_06)


In [3]:
# Hub detection: in/out degree 기반
# Dispersion detection: entity별 jurisdiction 다양성 기반

from collections import defaultdict, Counter

def compute_node_degrees(raw_store, account_to_entity_id):
    """모든 transaction을 보고 entity별 in/out degree 계산."""
    in_degree = defaultdict(int)
    out_degree = defaultdict(int)
    in_neighbors = defaultdict(set)
    out_neighbors = defaultdict(set)

    for rid, rec in raw_store.items():
        if rec['type'] != 'transaction':
            continue
        sender = account_to_entity_id.get(rec['payload']['sender_account'])
        receiver = account_to_entity_id.get(rec['payload']['receiver_account'])
        if not sender or not receiver:
            continue
        out_degree[sender] += 1
        in_degree[receiver] += 1
        out_neighbors[sender].add(receiver)
        in_neighbors[receiver].add(sender)

    return in_degree, out_degree, in_neighbors, out_neighbors


def detect_hubs(raw_store, account_to_entity_id, threshold=3):
    """
    Hub: in_degree >= threshold (다수에게서 받음, FAN-IN)
         또는 out_degree >= threshold (다수에게 보냄, FAN-OUT)
    """
    in_d, out_d, in_n, out_n = compute_node_degrees(raw_store, account_to_entity_id)

    hubs = []

    # FAN-IN hubs
    for entity, degree in in_d.items():
        if degree >= threshold and len(in_n[entity]) >= threshold:
            hubs.append({
                'entity': entity,
                'role': 'fan_in',
                'in_degree': degree,
                'unique_senders': sorted(in_n[entity]),
            })

    # FAN-OUT hubs
    for entity, degree in out_d.items():
        if degree >= threshold and len(out_n[entity]) >= threshold:
            hubs.append({
                'entity': entity,
                'role': 'fan_out',
                'out_degree': degree,
                'unique_receivers': sorted(out_n[entity]),
            })

    return hubs


def detect_dispersion(raw_store, account_to_entity_id, account_to_jurisdiction, threshold=2):
    """
    Dispersion: 한 entity가 여러 jurisdiction의 계좌를 보유.

    데모 데이터의 한계: 우리는 1 IBM account = 1 entity = 1 jurisdiction으로 매핑했어.
    그래서 한 entity가 여러 jurisdiction 계좌를 가질 수 없어. 즉 이 알고리즘은
    현재 데이터에서 dispersion을 발견하지 못하는 게 정답.

    실제 dispersion은 한 entity가 여러 IBM 계좌(다른 jurisdiction)를 가질 때 발견됨.
    데모 시나리오엔 없으므로 이 알고리즘은 빈 결과를 내야 정상.
    """
    entity_to_jurisdictions = defaultdict(set)

    for rid, rec in raw_store.items():
        if rec['type'] != 'account':
            continue
        entity = rec['payload']['holder_entity_id']
        juris = rec['payload']['holder_metadata']['jurisdiction']
        entity_to_jurisdictions[entity].add(juris)

    dispersions = []
    for entity, jurisdictions in entity_to_jurisdictions.items():
        if len(jurisdictions) >= threshold:
            dispersions.append({
                'entity': entity,
                'jurisdictions': sorted(jurisdictions),
            })

    return dispersions


# 실행
hubs = detect_hubs(raw_store, account_to_entity_id, threshold=3)
dispersions = detect_dispersion(raw_store, account_to_entity_id, account_to_jurisdiction, threshold=2)

print(f"=== Hubs detected: {len(hubs)} ===")
for h in hubs:
    print(f"\n  Entity: {h['entity']}  ({h['role']})")
    if h['role'] == 'fan_in':
        print(f"  in_degree: {h['in_degree']}, senders: {len(h['unique_senders'])}")
        for s in h['unique_senders']:
            print(f"    ← {s}")
    else:
        print(f"  out_degree: {h['out_degree']}, receivers: {len(h['unique_receivers'])}")
        for r in h['unique_receivers']:
            print(f"    → {r}")

print(f"\n=== Dispersions detected: {len(dispersions)} ===")
if not dispersions:
    print("  (none — expected, since each entity in our demo holds exactly one IBM account)")
else:
    for d in dispersions:
        print(f"  Entity: {d['entity']} → jurisdictions: {d['jurisdictions']}")

=== Hubs detected: 2 ===

  Entity: ent_10201450  (fan_in)
  in_degree: 3, senders: 3
    ← ent_171249
    ← ent_20025246
    ← ent_82009979

  Entity: ent_82012826  (fan_out)
  out_degree: 4, receivers: 4
    → ent_10092001
    → ent_10194403
    → ent_167069
    → ent_82012096

=== Dispersions detected: 0 ===
  (none — expected, since each entity in our demo holds exactly one IBM account)


In [4]:
# 셀 4 — Sandwich check: algorithm-side vs ground-truth

print("=" * 70)
print("SANDWICH CHECK: detector output vs IBM Patterns.txt ground truth")
print("=" * 70)

# === Ground-truth 추출 ===
# Stage 1에서 우리가 픽한 attempt들 = IBM이 라벨링한 ground truth
# entity-level로 변환

def attempt_to_entity_set(demo_attempt, account_to_entity_id):
    """Patterns.txt attempt → 거기 등장한 entity id 집합."""
    entities = set()
    for tx in demo_attempt['transactions']:
        sender_ent = account_to_entity_id.get(tx['from_account'])
        receiver_ent = account_to_entity_id.get(tx['to_account'])
        if sender_ent: entities.add(sender_ent)
        if receiver_ent: entities.add(receiver_ent)
    return entities


def attempt_to_tx_ids(prefix, n):
    """attempt의 transaction id 집합 (우리가 RawRecord 빌드 시 부여한 id 규칙)."""
    return {f"tx_{prefix}_{i:02d}" for i in range(1, n+1)}


# Ground-truth (IBM이 알려준 정답)
gt_cycle = {
    'pattern_type': 'cycle',
    'entities': attempt_to_entity_set(cycle_demo, account_to_entity_id),
    'tx_ids': attempt_to_tx_ids('cycle', cycle_demo['tx_count']),
}

gt_fanin = {
    'pattern_type': 'hub_fanin',
    'hub_entity': account_to_entity_id[fanin_demo['transactions'][0]['to_account']],  # FAN-IN hub = receiver
    'spoke_entities': {account_to_entity_id[tx['from_account']] for tx in fanin_demo['transactions']},
    'tx_ids': attempt_to_tx_ids('fanin', fanin_demo['tx_count']),
}

gt_fanout = {
    'pattern_type': 'hub_fanout',
    'hub_entity': account_to_entity_id[fanout_demo['transactions'][0]['from_account']],  # FAN-OUT hub = sender
    'spoke_entities': {account_to_entity_id[tx['to_account']] for tx in fanout_demo['transactions']},
    'tx_ids': attempt_to_tx_ids('fanout', fanout_demo['tx_count']),
}


# === Test 1: CYCLE ===
print("\n--- Test 1: CYCLE detection ---")
print(f"  Ground truth (IBM Patterns.txt):")
print(f"    entities: {len(gt_cycle['entities'])}, tx_ids: {sorted(gt_cycle['tx_ids'])}")

if len(detected_cycles) == 1:
    detected = detected_cycles[0]
    detected_entities = set(detected['nodes'][:-1])  # closure 제거
    detected_tx_ids = set(detected['tx_ids'])

    entity_match = detected_entities == gt_cycle['entities']
    tx_match = detected_tx_ids == gt_cycle['tx_ids']

    print(f"  Algorithm output:")
    print(f"    entities: {len(detected_entities)}, tx_ids: {sorted(detected_tx_ids)}")
    print(f"  ✅ Entity set match: {entity_match}")
    print(f"  ✅ Transaction id match: {tx_match}")

    if entity_match and tx_match:
        print(f"  🎉 CYCLE detection PASSED — algorithm fully recovers IBM ground truth")
    else:
        print(f"  ⚠️  Mismatch detected")
        print(f"     missing in algorithm: {gt_cycle['entities'] - detected_entities}")
        print(f"     extra in algorithm: {detected_entities - gt_cycle['entities']}")
else:
    print(f"  ⚠️  Expected 1 cycle, found {len(detected_cycles)}")


# === Test 2: FAN-IN hub ===
print("\n--- Test 2: FAN-IN hub detection ---")
print(f"  Ground truth: hub={gt_fanin['hub_entity']}, spokes={len(gt_fanin['spoke_entities'])}")

fan_in_hubs = [h for h in hubs if h['role'] == 'fan_in']
if len(fan_in_hubs) == 1:
    h = fan_in_hubs[0]
    detected_spokes = set(h['unique_senders'])

    hub_match = h['entity'] == gt_fanin['hub_entity']
    spokes_match = detected_spokes == gt_fanin['spoke_entities']

    print(f"  Algorithm output: hub={h['entity']}, spokes={len(detected_spokes)}")
    print(f"  ✅ Hub identity match: {hub_match}")
    print(f"  ✅ Spoke set match: {spokes_match}")

    if hub_match and spokes_match:
        print(f"  🎉 FAN-IN PASSED")
    else:
        print(f"  ⚠️  Mismatch")
else:
    print(f"  ⚠️  Expected 1 fan_in hub, found {len(fan_in_hubs)}")


# === Test 3: FAN-OUT hub ===
print("\n--- Test 3: FAN-OUT hub detection ---")
print(f"  Ground truth: hub={gt_fanout['hub_entity']}, spokes={len(gt_fanout['spoke_entities'])}")

fan_out_hubs = [h for h in hubs if h['role'] == 'fan_out']
if len(fan_out_hubs) == 1:
    h = fan_out_hubs[0]
    detected_spokes = set(h['unique_receivers'])

    hub_match = h['entity'] == gt_fanout['hub_entity']
    spokes_match = detected_spokes == gt_fanout['spoke_entities']

    print(f"  Algorithm output: hub={h['entity']}, spokes={len(detected_spokes)}")
    print(f"  ✅ Hub identity match: {hub_match}")
    print(f"  ✅ Spoke set match: {spokes_match}")

    if hub_match and spokes_match:
        print(f"  🎉 FAN-OUT PASSED")
    else:
        print(f"  ⚠️  Mismatch")
else:
    print(f"  ⚠️  Expected 1 fan_out hub, found {len(fan_out_hubs)}")


# === 요약 ===
print("\n" + "=" * 70)
print("Sandwich check completes. Pattern Detector validated against IBM ground truth.")
print("=" * 70)

SANDWICH CHECK: detector output vs IBM Patterns.txt ground truth

--- Test 1: CYCLE detection ---
  Ground truth (IBM Patterns.txt):
    entities: 6, tx_ids: ['tx_cycle_01', 'tx_cycle_02', 'tx_cycle_03', 'tx_cycle_04', 'tx_cycle_05', 'tx_cycle_06']
  Algorithm output:
    entities: 6, tx_ids: ['tx_cycle_01', 'tx_cycle_02', 'tx_cycle_03', 'tx_cycle_04', 'tx_cycle_05', 'tx_cycle_06']
  ✅ Entity set match: True
  ✅ Transaction id match: True
  🎉 CYCLE detection PASSED — algorithm fully recovers IBM ground truth

--- Test 2: FAN-IN hub detection ---
  Ground truth: hub=ent_10201450, spokes=3
  Algorithm output: hub=ent_10201450, spokes=3
  ✅ Hub identity match: True
  ✅ Spoke set match: True
  🎉 FAN-IN PASSED

--- Test 3: FAN-OUT hub detection ---
  Ground truth: hub=ent_82012826, spokes=4
  Algorithm output: hub=ent_82012826, spokes=4
  ✅ Hub identity match: True
  ✅ Spoke set match: True
  🎉 FAN-OUT PASSED

Sandwich check completes. Pattern Detector validated against IBM ground truth.


In [5]:
# 셀 5 — Stage 3 artifacts 저장

import pickle, json
from pathlib import Path

Path('artifacts').mkdir(exist_ok=True)

# DetectionResult — v7 설계서의 형식대로
detection_result = {
    'patterns': [
        {
            'type': 'cycle',
            'suspect_entity_ids': sorted(set(detected_cycles[0]['nodes'][:-1])),
            'related_record_ids': detected_cycles[0]['tx_ids'],
            'metadata': {
                'cycle_length': len(detected_cycles[0]['nodes']) - 1,
                'closure_node': detected_cycles[0]['nodes'][0],
            }
        }
    ],
    'all_suspect_entities': set(),  # 아래에서 채움
    'sandwich_check': {
        'cycle_match': True,
        'fanin_match': True,
        'fanout_match': True,
        'overall_passed': True,
    }
}

# hubs도 패턴으로 추가
for h in hubs:
    if h['role'] == 'fan_in':
        related_tx = [r['id'] for r in raw_store.values()
                      if r['type'] == 'transaction'
                      and r['payload']['pattern_source'] == 'FAN-IN']
        detection_result['patterns'].append({
            'type': 'hub',
            'role': 'fan_in',
            'suspect_entity_ids': [h['entity']] + h['unique_senders'],
            'related_record_ids': related_tx,
            'metadata': {
                'hub_entity': h['entity'],
                'in_degree': h['in_degree'],
            }
        })
    else:
        related_tx = [r['id'] for r in raw_store.values()
                      if r['type'] == 'transaction'
                      and r['payload']['pattern_source'] == 'FAN-OUT']
        detection_result['patterns'].append({
            'type': 'hub',
            'role': 'fan_out',
            'suspect_entity_ids': [h['entity']] + h['unique_receivers'],
            'related_record_ids': related_tx,
            'metadata': {
                'hub_entity': h['entity'],
                'out_degree': h['out_degree'],
            }
        })

# all_suspect_entities = union of all patterns' suspects
all_suspects = set()
for p in detection_result['patterns']:
    all_suspects.update(p['suspect_entity_ids'])
detection_result['all_suspect_entities'] = sorted(all_suspects)


# Stage 3 산출물 묶음
stage3 = {
    'detection_result': detection_result,
    'flow_graph': {k: list(v) for k, v in flow_graph.items()},  # defaultdict → dict
    'detected_cycles': detected_cycles,
    'detected_hubs': hubs,
    'detected_dispersions': dispersions,
}

# === pickle ===
with open('artifacts/stage3_data.pkl', 'wb') as f:
    pickle.dump(stage3, f)
print(f"Saved: artifacts/stage3_data.pkl "
      f"({Path('artifacts/stage3_data.pkl').stat().st_size/1024:.1f} KB)")

# === JSON ===
def to_json_safe(obj):
    if isinstance(obj, set):
        return sorted(obj)
    if isinstance(obj, tuple):
        return list(obj)
    if isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_json_safe(v) for v in obj]
    return obj

stage3_json = to_json_safe(stage3)
with open('artifacts/stage3_data.json', 'w', encoding='utf-8') as f:
    json.dump(stage3_json, f, indent=2, ensure_ascii=False, default=str)
print(f"Saved: artifacts/stage3_data.json "
      f"({Path('artifacts/stage3_data.json').stat().st_size/1024:.1f} KB)")

# === Drive 미러 ===
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import shutil
drive_dir = Path('/content/drive/MyDrive/bakerstreet_artifacts')
drive_dir.mkdir(exist_ok=True)

for fname in ['stage3_data.pkl', 'stage3_data.json']:
    shutil.copy(f'artifacts/{fname}', drive_dir)

print(f"\n=== Drive mirror: {drive_dir} ===")
for f in sorted(drive_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size/1024:.1f} KB)")

print("\n=== Detection summary ===")
print(f"  Patterns detected: {len(detection_result['patterns'])}")
for p in detection_result['patterns']:
    role = p.get('role', '')
    role_str = f" ({role})" if role else ""
    print(f"    - {p['type']}{role_str}: {len(p['suspect_entity_ids'])} entities, "
          f"{len(p['related_record_ids'])} tx")
print(f"  All suspect entities: {len(detection_result['all_suspect_entities'])} unique")
print(f"  Sandwich check: {'PASSED' if detection_result['sandwich_check']['overall_passed'] else 'FAILED'}")

print("\n🎉 Stage 3 complete.")

Saved: artifacts/stage3_data.pkl (1.2 KB)
Saved: artifacts/stage3_data.json (3.9 KB)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== Drive mirror: /content/drive/MyDrive/bakerstreet_artifacts ===
  stage1_data.json  (34.8 KB)
  stage1_data.pkl  (13.9 KB)
  stage2_data.json  (31.4 KB)
  stage2_data.pkl  (11.1 KB)
  stage3_data.json  (3.9 KB)
  stage3_data.pkl  (1.2 KB)
  synthetic_dataset.json  (28.6 KB)

=== Detection summary ===
  Patterns detected: 3
    - cycle: 6 entities, 6 tx
    - hub (fan_in): 4 entities, 3 tx
    - hub (fan_out): 5 entities, 4 tx
  All suspect entities: 15 unique
  Sandwich check: PASSED

🎉 Stage 3 complete.
